In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import sys

from google.colab import drive
drive.mount('/content/drive')

sys.path.append('/content/drive/MyDrive/wordle-solver-master/')
import deep_rl.wordle.wordle

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
class NN(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim,16),
            nn.Sigmoid(),
            nn.Linear(16,32),
            nn.Sigmoid(),
            nn.Linear(32,output_dim),
            nn.Softmax(dim=1)
        )
    def forward(self,x):
        return self.model(x)

class BaselineNN(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim,16),
            nn.ReLU(),
            nn.Linear(16,32),
            nn.ReLU(),
            nn.Linear(32,1)
        )
    def forward(self,x):
        return self.model(x)

In [ ]:
learning_rate = 1e-3
gamma = 0.99

seed = 12

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

env_name = "WordleEnv10-v0"
env = gym.make(env_name)

input_dim = env.observation_space.shape[0]
output_dim = env.action_space.n

model = NN(input_dim, output_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

baseline_model = BaselineNN(input_dim)
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=learning_rate)

episodes = 500000

all_rewards = []

record = 1000

for episode in range(episodes):
    state,_ = env.reset()
    state = torch.FloatTensor(state).unsqueeze(0)

    log_ps = []
    rewards = []
    values = []

    done = False
    while not done:
        action_p = model(state)
        dist = torch.distributions.Categorical(action_p)
        action = dist.sample()

        value = baseline_model(state)
        values = values + [value]

        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated

        log_p = dist.log_prob(action)


        log_ps.append(log_p)
        rewards = rewards + [reward]

        state = torch.FloatTensor(next_state).unsqueeze(0)

    losses = []
    baseline_losses = []
    T = len(rewards)

    if T == 1:
      all_rewards = all_rewards + [sum(rewards)]

      if episode % record == 0 and episode>0:
        mean_reward = sum(all_rewards[-record:]) / record
        print(mean_reward)
      continue
# I think there's an indexing error between our environment and the reinforce algorithm in the notes.
# As the environment is weighted with only rewards at the end, it ignores the rewards if written like the notes
# Removing the - 1 from the algorithm corrects the issue.
    for t in range(T):
        G = 0.0
        for k in range(t,T):
            G += (gamma**(k-t)) * rewards[k]
        loss = -log_ps[t] * (G - values[t].detach())
        losses = losses + [loss]
        baseline_loss = (G - values[t])**2
        baseline_losses = baseline_losses + [baseline_loss]

    final_loss = torch.stack(losses).mean()
    baseline_loss = torch.stack(baseline_losses).mean()

    optimizer.zero_grad()
    baseline_optimizer.zero_grad()
    final_loss.backward()
    baseline_loss.backward()
    optimizer.step()
    baseline_optimizer.step()

    all_rewards = all_rewards + [sum(rewards)]

    if episode % record == 0 and episode>0:
        mean_reward = sum(all_rewards[-record:]) / record
        print(mean_reward)



env.close()

-9.07
-8.9
-8.96
-8.98
-9.06
-9.17
-9.05
-8.76
-8.92
-8.87
-9.15
-8.82
-9.09
-9.31
-9.02
-9.06
-8.98
-8.97
-8.87
-8.93
-8.96
-8.82
-8.91
-9.02
-9.06
-8.79
-8.9
-8.97
-9.0
-8.94
-8.81
-8.89
-8.95
-8.71
-8.93
-9.09
-9.12
-8.75
-9.11
-8.64
-8.82
-8.83
-8.71
-8.68
-8.72
-8.96
-8.95
-8.95
-8.92
-8.47
-8.8
-8.75
-8.81
-8.78
-8.49
-8.85
-8.72
-8.89
-8.92
-8.72
-8.77
-8.66
-8.94
-8.66
-8.81
-8.55
-8.83
-8.82
-8.61
-8.8
-8.79
-8.62
-8.44
-8.33
-8.88
-8.51
-8.59
-8.53
-8.86
-8.75
-8.75
-8.9
-8.85
-8.82
-8.69
-8.4
-8.7
-8.56
-8.66
-8.77
-8.32
-8.67
-8.26
-8.5
-8.67
-8.37
-8.52
-8.49
-8.68
-8.57
-8.48
-8.65
-8.27
-8.18
-8.08
-8.38
-8.24
-8.05
-8.16
-7.81
-8.03
-8.34
-8.14
-8.24
-7.83
-8.23
-8.01
-8.03
-7.61
-8.05
-7.74
-7.82
-8.11
-7.78
-7.67
-7.94
-7.59
-7.62
-7.82
-7.44
-7.78
-7.64
-7.65
-7.49
-7.22
-7.37
-7.55
-7.22
-7.62
-7.18
-7.36
-7.56
-7.41
-7.3
-7.2
-7.29
-7.51
-7.51
-7.38
-7.41
-7.17
-7.69
-7.13
-7.41
-7.11
-7.38
-6.93
-6.89
-7.07
-7.0
-7.49
-7.41
-7.45
-7.59
-6.9
-6.97
-7.1
-7.29
-7.37


In [ ]:
print(len(all_rewards[-record:]))

100
